# Step 2 -- Online-pipeline validation (sub_001 / sub_002 / sub_003)

Does switching from best-case **offline** processing to the real-time
**causal/streaming** pipeline degrade decoding AUC? Deployment-faithful, paired
methodology: on the same functional-localizer trials and the same CV folds, fit
the decoder on **offline** train trials, then score the held-out trials twice --
once offline-processed (best-case reference + sanity anchor), once
online-processed (as-deployed). The offline/online AUC gap is attributable purely
to the pipeline (causal filtering, micro-batch boundary state, decimation phase);
the frozen ICA/interp matrices come from each subject's Phase-1 artifact and are
identical on both sides.

Every offline trial gets an online counterpart, so the CV partition matches step 1's
exactly (standard `StratifiedKFold`, all classes in train *and* test). Stimulus trials
pair to their online trigger marker; marker-less interval (rest) trials are epoched
from the continuous online feature stream at their own offline onset. The AUC therefore
matches step 1's rest-inflated number (rest separates easily from stimulus) -- read the
result as "the causal pipeline preserves decodability on step 1's partition" (the
work-plan bar is degradation on the *same data partition*), not as a category-
discrimination claim. The headline is the offline-vs-online *delta* on identical trials.

Per subject x decoder task, this collects:
- the cross-pipeline TGM (train-offline / test offline-and-online) + its diagonal,
  offline vs online (`online_validation.cross_pipeline_tgm`)
- the deployment operating-point degradation at the decoder's trained timepoint:
  paired AUC offline vs online, absolute/relative drop, bootstrap CI, pass/fail
  vs the 10% bar (`online_validation.operating_point_degradation`)

All outputs are written under `report_assets/step2/`. Per-subject engineering
validation only -- no cross-subject inference.

In [ ]:
%load_ext autoreload
%autoreload 2

from analysis_lib import context
REPO_ROOT = context.bootstrap()

from pathlib import Path

import joblib
import matplotlib
matplotlib.use("inline")
# Save every figure with a tight bounding box so titles/suptitles/labels drawn
# near or past the figure edge aren't cropped in PNGs.
matplotlib.rcParams["savefig.bbox"] = "tight"
matplotlib.rcParams["savefig.pad_inches"] = 0.1
import matplotlib.pyplot as plt
import mne
mne.set_log_level("ERROR")
import numpy as np

from analysis_lib import online_validation, plots, streaming

## Knobs

Each subject is a `SessionPaths`-shaped output dir with `epochs/` **and**
`models/decoder_pipeline.joblib` (the trained Phase-1 artifact -- required here,
since its frozen `online_state` drives the online pipeline). `BATCH_SIZE_SAMPLES`
replays the stream at the live StreamWorker cadence; `MAX_SECONDS=None` uses the
whole recording.

In [15]:
SUBJECTS = ["data/sub_001", "data/sub_002", "data/sub_003"]

BATCH_SIZE_SAMPLES = 40    # StreamWorker-style micro-batch cadence
MAX_SECONDS = None         # None = full recording
THRESHOLD = 0.10           # work-plan bar: <=10% relative AUC drop
N_BOOT = 2000              # bootstrap resamples for the degradation CI

ASSETS_DIR = REPO_ROOT / "tests" / "notebooks" / "analysis" / "report_assets" / "step2"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR

WindowsPath('C:/Users/itaip/projects/live-reactivation-decoder/tests/notebooks/analysis/report_assets/step2')

## Load contexts + resolve each subject's FL recording

`context.load_context` (not `load_offline_context`) -- step 2 needs the trained
decoder artifact for its `online_state` and trained timepoints. The FL recording
dir is discovered from `ctx.raw_dirs`; if a subject has several raw dirs, set its
key in `FL_DIR_KEY` below.

In [16]:
FL_DIR_KEY = {}  # {subject: raw_dir_key} override; else auto-pick when unambiguous


def resolve_fl_dir(ctx, subject):
    keys = list(ctx.raw_dirs)
    if subject in FL_DIR_KEY:
        return ctx.raw_dirs[FL_DIR_KEY[subject]]
    fl = [k for k in keys if "local" in k.lower() or "fl" in k.lower()]
    if len(fl) == 1:
        return ctx.raw_dirs[fl[0]]
    if len(keys) == 1:
        return ctx.raw_dirs[keys[0]]
    raise ValueError(f"{subject}: ambiguous raw dirs {keys} -- set FL_DIR_KEY[{subject!r}]")


ctxs = {s: context.load_context(s) for s in SUBJECTS}
fl_dirs = {s: resolve_fl_dir(ctxs[s], s) for s in SUBJECTS}

TASKS = ctxs[SUBJECTS[0]].settings.get_decoder_settings()["tasks"]
TASK_NAMES = [t["name"] for t in TASKS]
SUBJECT_NAMES = {s: Path(s).name for s in SUBJECTS}
{SUBJECT_NAMES[s]: fl_dirs[s].name for s in SUBJECTS}

{'sub_001': 'functinal_localizer',
 'sub_002': 'functional_localizer',
 'sub_003': 'functional_localizer'}

## Replay each FL recording once through the online pipeline

One `run_online_stream` pass per subject produces the multi-channel online
feature stream; the offline epochs are read once from `epochs/*epo.fif`. Both are
reused across the subject's decoder tasks below.

In [17]:
replay = {}  # subject -> dict(raw, features, out_samples, sfreq, fs_out, offline_epochs)
for s in SUBJECTS:
    ctx = ctxs[s]
    raw, eeg, sfreq = streaming.load_recording(fl_dirs[s], MAX_SECONDS)
    features, out_samples, fs_out = streaming.run_online_stream(
        ctx.preproc, eeg, batch_size=BATCH_SIZE_SAMPLES)
    offline_epochs = online_validation.load_offline_epochs(ctx)
    replay[s] = dict(raw=raw, features=features, out_samples=out_samples,
                     sfreq=sfreq, fs_out=fs_out, offline_epochs=offline_epochs)
    print(f"{SUBJECT_NAMES[s]}: features {features.shape}, "
          f"offline epochs {len(offline_epochs)}")

sub_001: features (274706, 64), offline epochs 604
sub_002: features (273549, 64), offline epochs 604
sub_003: features (303091, 64), offline epochs 604


## Cross-pipeline CV per subject/task

For each decoder: pair every offline trial to an online epoch (stimulus trials via
their trigger marker, interval/rest trials via their offline onset), then run the
cross-pipeline TGM and the paired operating-point degradation with standard
`StratifiedKFold` -- all classes in train and test, matching step 1's partition.
`n_times` bounds online markers to the replayed span so a cropped `MAX_SECONDS`
stays consistent.

In [18]:
tgm_results = {}   # (subject, task) -> cross_pipeline_tgm dict (+ t_grid)
deg_results = {}   # (subject, task) -> operating_point_degradation dict (+ tp)

for s in SUBJECTS:
    ctx = ctxs[s]
    settings = ctx.settings.get_decoder_settings()
    r = replay[s]
    n_times = r["features"].shape[0] and int(r["out_samples"].max()) + 1
    for task_cfg in settings["tasks"]:
        name = task_cfg["name"]
        paired = online_validation.build_paired_features(
            ctx, r["offline_epochs"], task_cfg, r["raw"],
            r["features"], r["out_samples"], r["sfreq"], r["fs_out"],
            tmin=ctx.epoch_tmin, tmax=ctx.epoch_tmax, n_times=n_times)

        res = online_validation.cross_pipeline_tgm(
            paired["X_off"], paired["X_on"], paired["y"], settings)
        res["t_grid"] = paired["t_grid"]
        tgm_results[(s, name)] = res

        tp = ctx.task_tp(name)
        if tp is None:  # no trained tp -> fall back to the offline diagonal peak
            tp = float(paired["t_grid"][int(np.argmax(res["diag_off"]))])
        ti = int(np.argmin(np.abs(paired["t_grid"] - tp)))
        deg = online_validation.operating_point_degradation(
            paired["X_off"], paired["X_on"], paired["y"],
            settings, ti, n_boot=N_BOOT, threshold=THRESHOLD, rng=np.random.default_rng(0))
        deg["tp"] = tp
        deg["n_trials"] = paired["n_trials"]
        deg["n_marker"] = paired["n_marker"]
        deg["n_intervalied"] = paired["n_intervalied"]
        deg_results[(s, name)] = deg

        lo, hi = deg["ci"]
        print(f"{SUBJECT_NAMES[s]:>10s} | {name:<18s} @ {tp:+.2f}s | "
              f"trials {paired['n_trials']:3d} (marker {paired['n_marker']}, "
              f"interval {paired['n_intervalied']}) | "
              f"AUC off {deg['auc_off']:.3f} -> on {deg['auc_on']:.3f} | "
              f"drop {deg['pct_drop']*100:+5.1f}% [{lo*100:+.1f}, {hi*100:+.1f}] | "
              f"{'PASS' if deg['passed'] else 'FAIL'}")

   sub_001 | animate decoder    @ +0.19s | trials 600 (marker 300, interval 300) | AUC off 0.711 -> on 0.684 | drop  +3.8% [-2.0, +9.9] | PASS
   sub_001 | inanimate decoder  @ +0.17s | trials 600 (marker 300, interval 300) | AUC off 0.746 -> on 0.681 | drop  +8.8% [+2.2, +14.6] | PASS
   sub_002 | animate decoder    @ +0.27s | trials 600 (marker 300, interval 300) | AUC off 0.730 -> on 0.634 | drop +13.1% [+7.4, +19.0] | FAIL
   sub_002 | inanimate decoder  @ +0.33s | trials 600 (marker 300, interval 300) | AUC off 0.793 -> on 0.711 | drop +10.3% [+5.2, +15.9] | FAIL
   sub_003 | animate decoder    @ +0.21s | trials 600 (marker 300, interval 300) | AUC off 0.864 -> on 0.785 | drop  +9.1% [+5.2, +13.0] | PASS
   sub_003 | inanimate decoder  @ +0.17s | trials 600 (marker 300, interval 300) | AUC off 0.756 -> on 0.696 | drop  +7.9% [+2.4, +12.9] | PASS


## Per-subject degradation table

One row per subject x decoder: trained timepoint, offline vs online AUC, absolute
and relative drop, bootstrap CI on the relative drop, pass/fail vs the 10% bar.
Saved as `step2_results.csv`.

In [19]:
import csv

FIELDS = ["subject", "task", "tp", "auc_off", "auc_on", "delta",
          "pct_drop", "ci_lo", "ci_hi", "passed", "n_trials", "n_intervalied"]
rows = []
for s in SUBJECTS:
    for name in TASK_NAMES:
        d = deg_results[(s, name)]
        rows.append({
            "subject": SUBJECT_NAMES[s], "task": name, "tp": d["tp"],
            "auc_off": d["auc_off"], "auc_on": d["auc_on"], "delta": d["delta"],
            "pct_drop": d["pct_drop"], "ci_lo": d["ci"][0], "ci_hi": d["ci"][1],
            "passed": d["passed"], "n_trials": d["n_trials"], "n_intervalied": d["n_intervalied"],
        })

with open(ASSETS_DIR / "step2_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDS)
    writer.writeheader()
    writer.writerows(rows)

w = 11
header = ("subject".ljust(12) + "task".ljust(20)
         + "".join(h.rjust(w) for h in ("tp", "auc_off", "auc_on", "drop%", "pass")))
print(header); print("-" * len(header))
for r in rows:
    print(r["subject"].ljust(12) + r["task"].ljust(20)
          + f"{r['tp']:+.2f}".rjust(w) + f"{r['auc_off']:.3f}".rjust(w)
          + f"{r['auc_on']:.3f}".rjust(w) + f"{r['pct_drop']*100:+.1f}".rjust(w)
          + ("PASS" if r["passed"] else "FAIL").rjust(w))

subject     task                         tp    auc_off     auc_on      drop%       pass
---------------------------------------------------------------------------------------
sub_001     animate decoder           +0.19      0.711      0.684       +3.8       PASS
sub_001     inanimate decoder         +0.17      0.746      0.681       +8.8       PASS
sub_002     animate decoder           +0.27      0.730      0.634      +13.1       FAIL
sub_002     inanimate decoder         +0.33      0.793      0.711      +10.3       FAIL
sub_003     animate decoder           +0.21      0.864      0.785       +9.1       PASS
sub_003     inanimate decoder         +0.17      0.756      0.696       +7.9       PASS


## Offline vs online diagonal AUC over time

The single operating-point number can hide a *latency shift* -- causal filtering
may push the peak later rather than lower. Overlaying the offline (solid) and
online (dashed) diagonal-AUC curves exposes that; the trained timepoint is
marked.

In [ ]:
for s in SUBJECTS:
    name_s = SUBJECT_NAMES[s]
    fig, axes = plt.subplots(1, len(TASK_NAMES), figsize=(6 * len(TASK_NAMES), 4.2), squeeze=False)
    for idx, task in enumerate(TASK_NAMES):
        ax = axes[0][idx]
        res = tgm_results[(s, task)]
        tg = res["t_grid"]
        ax.plot(tg, res["diag_off"], color="darkgreen", lw=2.0, label="offline")
        ax.plot(tg, res["diag_on"], color="navy", lw=2.0, ls="--", label="online")
        ax.axhline(0.5, color="gray", ls="--", lw=0.8)
        ax.axvline(0.0, color="k", ls=":", lw=0.8)
        ax.set(title=f"{name_s} -- {task}", ylim=(0.4, 1.0),
               xlabel="time (s)", ylabel="diagonal AUC")
        ax.legend(fontsize=7, loc="lower right")
    plt.tight_layout()
    fig.savefig(ASSETS_DIR / f"diag_auc_{name_s}.png", dpi=150, bbox_inches="tight")
    plt.show()

## Cross-pipeline TGM heatmaps

Train-offline-time x test-online-time AUC. The diagonal is the deployed
(train==test time) case; off-diagonal structure reveals where the causal pipeline
shifts the readout in time.

In [ ]:
for s in SUBJECTS:
    name_s = SUBJECT_NAMES[s]
    for task in TASK_NAMES:
        res = tgm_results[(s, task)]
        tg = res["t_grid"]
        fig, ax = plt.subplots(figsize=(5, 4.5))
        im = plots.plot_tgm_matrix(res["tgm_on"], tg, f"{name_s} -- {task} (online test)", ax=ax)
        fig.colorbar(im, ax=ax, fraction=0.046, label="AUC")
        plt.tight_layout()
        fig.savefig(ASSETS_DIR / f"tgm_online_{name_s}_{task.replace(' ', '_')}.png", dpi=150, bbox_inches="tight")
        plt.show()

## Save summary

Bundles the TGMs, diagonals, degradation results, and the table into one joblib
for reuse when writing the report.

In [ ]:
summary = {
    "subjects": SUBJECTS,
    "tasks": TASK_NAMES,
    "threshold": THRESHOLD,
    "batch_size": BATCH_SIZE_SAMPLES,
    "tgm_results": tgm_results,
    "deg_results": deg_results,
    "results_table": rows,
}
out_path = ASSETS_DIR / "step2_summary.joblib"
joblib.dump(summary, out_path)
print(f"saved to {out_path}")